In [1]:
from solvers import anderson, broyden
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
sys.path.append('C:/Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils')

from NN_utils import *
import torch
import torch.nn as nn
from torchvision import datasets, transforms
#from torchsummary import summary
import time
from types import SimpleNamespace
import pickle
import gc
from torch.utils.data import DataLoader

from optimization_algorithms import *

from torch.func import functional_call
from torch import vmap
from collections import OrderedDict


In [2]:
# --- Parameters ---
N_in = 10
N_neurons = 64
alpha = 3.0
dt = 0.1
batch_size = 1
threshold = 50
eps = 1e-4

# --- Create input ---
x = torch.randn(batch_size, N_in)

# --- Initialize weights ---
W_input = torch.randn(N_neurons, N_in) * 0.1
W_recurrent = torch.randn(N_neurons, N_neurons)
# Spectral normalization
eigvals = torch.linalg.eigvals(W_recurrent).abs().max()
W_recurrent *= 0.8 / eigvals

# Project input once
x_proj = F.linear(x, W_input)  # shape: (1, N_neurons)

# --- Define function f(h) = h + dt * (-alpha * h + sin(x_proj + W_rec h)) ---
def f(h):
    h_flat = h.squeeze(-1)  # shape (B, D)
    preact = x_proj + F.linear(h_flat, W_recurrent)  # shape (B, D)
    dh = -alpha * h_flat + torch.sin(preact)
    return (h_flat + dt * dh).unsqueeze(-1)

# --- Initial guess for h ---
h0 = torch.zeros(batch_size, N_neurons)
h0 = h0.unsqueeze(-1)  # Anderson expects shape (B, D, L), here L=1

# --- Call the Anderson solver ---
start_time = time.time()
result = anderson(f, h0, threshold=threshold, eps=eps, stop_mode='rel')
end_time = time.time()
# --- Display the results ---
h_star = result['result'].squeeze(-1)
print("Converged in", result['nstep'], "steps")
print("Final residual norm:", result['lowest'])
print("Steady state h* shape:", h_star.shape)

Converged in 49 steps
Final residual norm: 0.00020069324817258543
Steady state h* shape: torch.Size([1, 64])


In [3]:
# --- Parameters ---
N_in = 10
N_neurons = 64
alpha = 3.0
dt = 0.1
batch_size = 1
threshold = 50
eps = 1e-4

# --- Create input ---
x = torch.randn(batch_size, N_in)

# --- Initialize weights ---
W_input = torch.randn(N_neurons, N_in) * 0.1
W_recurrent = torch.randn(N_neurons, N_neurons)
# Spectral normalization
eigvals = torch.linalg.eigvals(W_recurrent).abs().max()
W_recurrent *= 0.8 / eigvals

# Project input once
x_proj = F.linear(x, W_input)  # shape: (1, N_neurons)

# --- Define function f(h) = h + dt * (-alpha * h + sin(x_proj + W_rec h)) ---
def f(h):
    h_flat = h.squeeze(-1)  # shape (B, D)
    preact = x_proj + F.linear(h_flat, W_recurrent)
    dh = -alpha * h_flat + torch.sin(preact)
    return (h_flat + dt * dh).unsqueeze(-1)

# --- Initial guess for h ---
h0 = torch.zeros(batch_size, N_neurons).unsqueeze(-1)  # shape (B, D, L=1)

# --- Call the Broyden solver ---
start_time = time.time()
result = broyden(f, h0, threshold=threshold, eps=eps, stop_mode='rel')
end_time = time.time()

print(f'Time to steady state: {end_time - start_time} s')
# --- Display the results ---
h_star = result['result'].squeeze(-1)
print("Converged in", result['nstep'], "steps")
print("Final residual norm:", result['lowest'])
print("Steady state h* shape:", h_star.shape)

Time to steady state: 0.02985405921936035 s
Converged in 9 steps
Final residual norm: 8.164855111177376e-05
Steady state h* shape: torch.Size([1, 64])


In [2]:
# --- Parameters ---
RNN_params = {
    "N_in": 784,
    "N_out": 10,
    "N_neurons": 100,
    "N_layers": 3,
}


model = Oscillator_RNN_parallel_DEQ(params=RNN_params)
model.init_esn_weights(reservoir=True)

model.save_activations = True
# Set model to eval mode and CPU (or use GPU if preferred)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- Load one FashionMNIST sample ---
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
# Create DataLoader for a batch of 100 samples
loader = DataLoader(dataset, batch_size=1000, shuffle=False)

# Get one batch
img, label = next(iter(loader))
# Flatten and prepare repeated input
x = img.to(device)  # (1, 784)

#model = torch.compile(model)
# --- Forward Pass ---
start_time = time.time()
with torch.no_grad():
    y_pred = model.forward_euler(x)
end_time = time.time()


print(f' Regular forward pass time euler {end_time - start_time}')

model.solver = 'anderson'
model.eps_int = 1e-4
# --- Forward Pass ---
start_time = time.time()
with torch.no_grad():
    y_pred = model.forward_fixed_point_all_layers(x)
end_time = time.time()


print(f' DEQ solver forward pass time {end_time - start_time}')


model.solver = 'anderson'
model.eps_int = 1e-4
# --- Forward Pass ---
start_time = time.time()
with torch.no_grad():
    y_pred = model.forward(x)
end_time = time.time()


print(f' DEQ global fixed point solver forward pass time {end_time - start_time}')

 Regular forward pass time euler 0.303330659866333
 DEQ solver forward pass time 0.5868222713470459
 DEQ global fixed point solver forward pass time 0.29561281204223633


In [16]:
def stack_activations_dict(activations_dict):
    """
    Convert {layer: [T tensors]} -> {layer: Tensor of shape (T, B, D)}
    """
    return {
        k: torch.stack(v_list, dim=0)  # (T, B, D)
        for k, v_list in activations_dict.items()
    }

In [ ]:
model = Oscillator_RNN_parallel_DEQ(params=RNN_params)
model.init_esn_weights(reservoir=True)

model.save_activations = True
# Set model to eval mode and CPU (or use GPU if preferred)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



y_pred = model(x)

euler_activations = model.activations.clone().cpu()

model.solver = 'anderson'
model.eps_int = 1e-4


y_pred = model.forward_fixed_point_all_layers(x)

layer_fp_activations = stack_activations_dict(model.activations)

y_pred = model.forward_global_fixed_point(x)

global_fp_activations = stack_activations_dict(model.activations)




In [15]:
print(np.shape(rnn_out['layer2']))

(1, 1000, 100)


In [ ]:
print((torch.stack(global_fp_activations))

{'layer0': tensor([[[ 0.0042,  0.0433, -0.0495,  ..., -0.0088, -0.0401,  0.0484],
         [ 0.0554,  0.0221, -0.0149,  ..., -0.0252, -0.0465,  0.0271],
         [ 0.0270, -0.0001, -0.0198,  ..., -0.0109, -0.0366,  0.0159],
         ...,
         [ 0.0422,  0.0023, -0.0542,  ..., -0.0104, -0.0681,  0.0160],
         [ 0.0419,  0.0250, -0.0818,  ..., -0.0226, -0.0990,  0.0315],
         [ 0.0558,  0.0150, -0.0434,  ..., -0.0129, -0.0441,  0.0344]]]), 'layer1': tensor([[[ 0.0077, -0.0162,  0.0563,  ...,  0.0150, -0.0079, -0.0272],
         [ 0.0015, -0.0112,  0.0481,  ...,  0.0146, -0.0119, -0.0354],
         [ 0.0114, -0.0094,  0.0514,  ...,  0.0154, -0.0084, -0.0308],
         ...,
         [ 0.0074, -0.0099,  0.0555,  ...,  0.0216,  0.0006, -0.0352],
         [ 0.0057, -0.0120,  0.0563,  ...,  0.0310, -0.0043, -0.0351],
         [ 0.0149, -0.0135,  0.0540,  ...,  0.0139, -0.0036, -0.0400]]]), 'layer2': tensor([[[ 0.0244,  0.0036, -0.0123,  ...,  0.0155, -0.0289,  0.0438],
         [ 0

In [ ]:
layer = 'layer2'
global_fp_activations[layer] - layer_fp_activations[layer]

tensor([[[-1.2023e-04,  1.6611e-04,  2.0037e-05,  ..., -4.4835e-04,
           3.4906e-06,  3.6534e-05],
         [-6.8212e-05,  6.7154e-05,  3.2024e-05,  ..., -3.7045e-04,
          -8.0982e-05,  4.9487e-05],
         [-1.0230e-04,  1.2460e-04,  7.5157e-05,  ..., -4.1933e-04,
           1.4376e-05,  1.7583e-05],
         ...,
         [-3.0182e-05,  1.7121e-04, -8.4825e-06,  ..., -3.5674e-04,
           4.4901e-05,  3.4057e-05],
         [-1.1296e-04,  9.7371e-05, -2.3588e-05,  ..., -3.9149e-04,
           7.4470e-05,  1.2125e-04],
         [-9.9298e-06,  1.1887e-04,  1.6258e-05,  ..., -4.4943e-04,
           4.5381e-05,  1.2781e-04]]])

In [30]:
euler_activations.shape

torch.Size([3, 40, 1000, 100])

In [37]:
torch.mean(euler_activations[1,-1,:,:] - global_fp_activations['layer1'])

tensor(-2.2500e-05)

In [3]:
n_neurons = 100

RNN_params = {
        "N_in": 784,               # e.g., flattened 28x28 FashionMNIST image
        "N_out": 10,               # number of classes in FashionMNIST
        "N_neurons": 100,          # number of hidden units per RNN layer
        "N_layers": 3,             # depth of the RNN
    }
    
model = Oscillator_RNN_parallel_DEQ(params=RNN_params).float()
    
model.init_esn_weights(reservoir = True)

model.save_activations = False

N_dim = model.count_parameters()



init_pos = model.get_params()

if init_pos.requires_grad:
    # Detach the tensor from the computation graph
    init_pos = init_pos.detach()
if init_pos.is_cuda:
    # Move the tensor to the CPU
    init_pos = init_pos.cpu()
init_pos = init_pos.numpy()

pop_size = int(0.01*N_dim)
PEPG_optimizer = PEPG_opt(N_dim, pop_size = pop_size, learning_rate=0.01, starting_mu=init_pos ,starting_sigma=1e-1)

PEPG_optimizer.sigma_decay = 0.9999
PEPG_optimizer.sigma_alpha=0.2
PEPG_optimizer.sigma_limit=0.01
PEPG_optimizer.elite_ratio=0.1
PEPG_optimizer.weight_decay=0.005


In [4]:
coordinates = PEPG_optimizer.ask()

In [5]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)

# Make functional and move buffers/params to device


# Vectorized model
batched_forward = vmap(
    lambda state, x: functional_call(model, state, (x,)),
    in_dims=(0, None),
)

# Move input to device
# Get one batch
img, label = next(iter(loader))
# Flatten and prepare repeated input
x = img.to(device)  # (1, 784)

# Build batched parameters on GPU
start_time = time.time()
# Forward pass on GPU
with torch.no_grad():
    
    batched_params = build_batched_params_dict_fast(coordinates, model, dtype=torch.float32, device=device)

    Y_pred_parallel = batched_forward(batched_params, x)

end_time = time.time()
print(f'Parallel forward pass time: {end_time - start_time:.4f} seconds')

Using device: cuda


RuntimeError: vmap: inplace arithmetic(self, *extra_args) is not possible because there exists a Tensor `other` in extra_args that has more elements than `self`. This happened due to `other` being vmapped over but `self` not being vmapped over in a vmap. Please try to use out-of-place operators instead of inplace arithmetic. If said operator is being called inside the PyTorch framework, please file a bug report instead.